# VertiRTA-X8-240 — 01. Offline 4D JAX HJI-CBVF & 2D CEL LUT Solver

**Recommended GPU:** **Google Colab `L4 (24 GB)`** (~5 min) | **Fallback:** **`T4 (16 GB)`** (~14 min).
**Do NOT allocate `G4 (96 GB)`:** Peak VRAM for the 2:1 nested `81x57x81x57` FP32 WENO5 + TVD-RK3 solve is **`< 6 GB`**; using `G4` wastes 3x–5x Compute Units with zero numerical benefit.

### Exact Artifacts Produced
1. `vertirta_x8_240_jax_v1.vrta_lut` — exact **`11,313,864 B`** (`4,096 B` header + `4 x 41x29x41x29` `int16` payload, stride-2 decimated from `81x57x81x57` with exact 21.3M-node `eps_grid_f` audit).
2. `vertirta_x8_240_cel2d_v1.vrta_cel2d` — exact **`11,230 B`** (`4,096 B` header + `3 x 41x29` `int16` payload across `T/W in {1.25, 1.20, 1.15}` HV battery sag tiers).
3. Full DR-2 Errata Lock & soundness verification via `reachability/verify_lut_soundness.py` + Drive backup with SHA-256 checksums.

In [ ]:
import subprocess

out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
    text=True,
).strip()
name, driver_ver, mem_mb_str = [part.strip() for part in out.splitlines()[0].split(",")]
vram_gb = float(mem_mb_str) / 1024.0
print(f"Detected GPU: {name} | Driver: {driver_ver} | VRAM: {vram_gb:.1f} GB")

if vram_gb >= 60.0:
    print(
        "[COST WARNING] You attached a >=60 GB GPU (e.g., G4 96 GB / A100 80 GB). "
        "This 4D HJI-CBVF solve peaks < 6 GB VRAM and runs cleanly on L4 (24 GB) or T4 (16 GB). "
        "Switch to L4/T4 to save Colab Compute Units!"
    )
elif vram_gb < 12.0:
    raise RuntimeError(f"Expected >= 12 GB CUDA GPU (T4/L4), detected {vram_gb:.1f} GB.")
else:
    print("GPU tier right-sized for VertiRTA 4D CBVF + 2D CEL LUT generation (L4 primary / T4 fallback).")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Google Colab; skipping drive.mount().")

REPO_URL = os.environ.get("VERTIRTA_REPO_URL", "https://github.com/vertirta/vertirta.git")
REPO_DIR = Path("/content/vertirta")
DRIVE_OUT_DIR = Path("/content/drive/MyDrive/VertiRTA_Artifacts/luts")
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Workspace ready:", Path.cwd())

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "jax[cuda12]",
        "hj-reachability>=0.7.0",
        "pyyaml",
        "numpy",
    ],
    check=True,
)

import jax
from reachability.cbvf_4d_dynamics import HAS_HJ_REACHABILITY, HAS_JAX

devices = jax.devices()
print("JAX devices:", devices)
assert HAS_JAX and HAS_HJ_REACHABILITY, "JAX + hj_reachability v0.7.0 must be importable!"
assert any(d.platform == "gpu" for d in devices), "Expected CUDA GPU backend for JAX!"

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

WORK_OUT = Path("/content/vertirta/results/luts")
WORK_OUT.mkdir(parents=True, exist_ok=True)
LUT_PATH = WORK_OUT / "vertirta_x8_240_jax_v1.vrta_lut"

t0 = time.perf_counter()
subprocess.run(
    [
        sys.executable,
        "-m",
        "reachability.solve_cbvf_luts",
        "--out",
        str(LUT_PATH),
        "--config",
        "config/vehicle_x8_240.yaml",
        "--horizon-s",
        "6.0",
    ],
    check=True,
)
elapsed_s = time.perf_counter() - t0
print(f"4D JAX 81x57x81x57 -> 41x29x41x29 solve finished in {elapsed_s:.1f} s ({LUT_PATH.stat().st_size:,} B)")

In [ ]:
import subprocess
import sys
from pathlib import Path

CEL_PATH = Path("/content/vertirta/results/luts/vertirta_x8_240_cel2d_v1.vrta_cel2d")
subprocess.run(
    [
        sys.executable,
        "-m",
        "reachability.solve_cel_2d_lut",
        "--out",
        str(CEL_PATH),
        "--config",
        "config/vehicle_x8_240.yaml",
    ],
    check=True,
)
print(f"2D CEL LUT exported: {CEL_PATH} ({CEL_PATH.stat().st_size:,} B)")

In [ ]:
import subprocess
import sys
from pathlib import Path

LUT_PATH = Path("/content/vertirta/results/luts/vertirta_x8_240_jax_v1.vrta_lut")
CEL_PATH = Path("/content/vertirta/results/luts/vertirta_x8_240_cel2d_v1.vrta_cel2d")

subprocess.run(
    [
        sys.executable,
        "reachability/verify_lut_soundness.py",
        "--lut",
        str(LUT_PATH),
        "--cel2d",
        str(CEL_PATH),
    ],
    check=True,
)

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

from reachability.solve_cbvf_luts import load_vrta_lut
from reachability.solve_cel_2d_lut import load_cel_2d_lut

WORK_OUT = Path("/content/vertirta/results/luts")
DRIVE_OUT = Path("/content/drive/MyDrive/VertiRTA_Artifacts/luts")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

lut_src = WORK_OUT / "vertirta_x8_240_jax_v1.vrta_lut"
cel_src = WORK_OUT / "vertirta_x8_240_cel2d_v1.vrta_cel2d"
lut_hdr, _ = load_vrta_lut(lut_src, verify_sha256=True)
cel_hdr, _ = load_cel_2d_lut(cel_src, verify_sha256=True)

manifest = {
    "vrta_lut": {
        "filename": lut_src.name,
        "bytes": lut_src.stat().st_size,
        "file_sha256": hashlib.sha256(lut_src.read_bytes()).hexdigest(),
        "payload_sha256": lut_hdr.payload_sha256.hex(),
        "eps_grid_m": list(lut_hdr.eps_grid),
        "v_cap_f_mps": list(lut_hdr.v_cap_f),
        "is_quick_fixture": lut_hdr.is_quick_fixture,
    },
    "vrta_cel2d": {
        "filename": cel_src.name,
        "bytes": cel_src.stat().st_size,
        "file_sha256": hashlib.sha256(cel_src.read_bytes()).hexdigest(),
        "payload_sha256": cel_hdr.payload_sha256.hex(),
        "eps_grid_m": list(cel_hdr.eps_grid),
        "a_net_tiers_mps2": list(cel_hdr.a_net_tiers),
    },
}

for src in (lut_src, cel_src):
    shutil.copy2(src, DRIVE_OUT / src.name)
(DRIVE_OUT / "lut_checksums.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(json.dumps(manifest, indent=2))